# HDB Resale Price Regression — Notebook 9: Interaction Terms

Testing whether the effect of some variables depends on another variable:

- **storey × town** — does a high floor matter more in some towns than others?
- **remaining_lease × town** — does lease decay hurt more in less desirable towns?
- **floor_area_sqm × dist_cbd_km** — is extra space worth more in central areas?

In [1]:
%load_ext rpy2.ipython
import warnings
warnings.filterwarnings('ignore')

Error importing in API mode: ImportError("dlopen(/Users/wongpeiting/.pyenv/versions/3.13.9/lib/python3.13/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): Library not loaded: /Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib\n  Referenced from: <B96A8100-FA7A-3EFC-8726-931D26646DE6> /Users/wongpeiting/.pyenv/versions/3.13.9/lib/python3.13/site-packages/_rinterface_cffi_api.abi3.so\n  Reason: tried: '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file)")


Trying to import in ABI mode.


In [2]:
%%R
library(tidyverse)
library(sandwich)
library(lmtest)

df <- read_csv('data/hdb_analysis.csv', show_col_types = FALSE)
df$remaining_lease_sq <- df$remaining_lease_years^2
df$month_factor <- factor(format(df$month, '%Y-%m'))

cat(sprintf('Loaded %s rows\n', format(nrow(df), big.mark = ',')))

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.2
✔ purrr     1.2.1     


── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


Loaded 51,748 rows


Loading required package: zoo

Attaching package: ‘zoo’

The following objects are masked from ‘package:base’:

    as.Date, as.Date.numeric



## Baseline: Model 10 from Notebook 6

In [3]:
%%R
model10 <- lm(resale_price ~ town + flat_type + floor_area_sqm + storey_mid +
              remaining_lease_years + remaining_lease_sq +
              flat_model_grouped +
              dist_cbd_km + mrt_dist_m + hawker_dist_m +
              popular_school_dist_m +
              park_dist_m + hospital_dist_m +
              columbarium_dist_m + temple_dist_m +
              coast_dist_m +
              num_eights_tail +
              price_has_168 +
              block_has_4 +
              cny_month +
              month_factor,
            data = df)

cat(sprintf('Model 10 (baseline)\n'))
cat(sprintf('R-squared:     %.4f\n', summary(model10)$r.squared))
cat(sprintf('Adj R-squared: %.4f\n', summary(model10)$adj.r.squared))

Model 10 (baseline)


R-squared:     0.9023


Adj R-squared: 0.9021


## Interaction 1: storey × town

Does a high floor matter more in some towns? A high floor in Bukit Timah (landed area, views of greenery) might be worth more than a high floor in Sengkang (all HDB, nothing to see).

In [4]:
%%R
model_int1 <- lm(resale_price ~ town * storey_mid + flat_type + floor_area_sqm +
              remaining_lease_years + remaining_lease_sq +
              flat_model_grouped +
              dist_cbd_km + mrt_dist_m + hawker_dist_m +
              popular_school_dist_m +
              park_dist_m + hospital_dist_m +
              columbarium_dist_m + temple_dist_m +
              coast_dist_m +
              num_eights_tail +
              price_has_168 +
              block_has_4 +
              cny_month +
              month_factor,
            data = df)

cat(sprintf('Model 10 + storey × town\n'))
cat(sprintf('R-squared:     %.4f (baseline: %.4f, improvement: %+.4f)\n',
    summary(model_int1)$r.squared, summary(model10)$r.squared,
    summary(model_int1)$r.squared - summary(model10)$r.squared))

cat('\nANOVA: does the interaction significantly improve the model?\n')
print(anova(model10, model_int1))

# Show the interaction coefficients
cat('\nInteraction coefficients (how much MORE each floor is worth in each town vs Ang Mo Kio):\n')
robust <- coeftest(model_int1, vcov = vcovHC(model_int1, type = 'HC1'))
int_rows <- grep('town.*:storey_mid', rownames(robust))
for (i in int_rows) {
    town_name <- gsub('town|:storey_mid', '', rownames(robust)[i])
    coef_val <- robust[i, 'Estimate']
    p_val <- robust[i, 'Pr(>|t|)']
    sig <- ifelse(p_val < 0.001, '***', ifelse(p_val < 0.01, '**', ifelse(p_val < 0.05, '*', '')))
    cat(sprintf('  %-20s: %+6.0f per floor %s (p = %.4f)\n', town_name, coef_val, sig, p_val))
}

Model 10 + storey × town


R-squared:     0.9065 (baseline: 0.9023, improvement: +0.0042)



ANOVA: does the interaction significantly improve the model?


Analysis of Variance Table


Model 1: resale_price ~ town + flat_type + floor_area_sqm + storey_mid + 
    remaining_lease_years + remaining_lease_sq + flat_model_grouped + 
    dist_cbd_km + mrt_dist_m + hawker_dist_m + popular_school_dist_m + 
    park_dist_m + hospital_dist_m + columbarium_dist_m + temple_dist_m + 
    coast_dist_m + num_eights_tail + price_has_168 + block_has_4 + 
    cny_month + month_factor
Model 2: resale_price ~ town * storey_mid + flat_type + floor_area_sqm + 
    remaining_lease_years + remaining_lease_sq + flat_model_grouped + 
    dist_cbd_km + mrt_dist_m + hawker_dist_m + popular_school_dist_m + 
    park_dist_m + hospital_dist_m + columbarium_dist_m + temple_dist_m + 
    coast_dist_m + num_eights_tail + price_has_168 + block_has_4 + 
    cny_month + month_factor

...

  SERANGOON           :  -2643 per floor *** (p = 0.0000)


  TAMPINES            :  -4267 per floor *** (p = 0.0000)


  TOA PAYOH           :  -1498 per floor *** (p = 0.0002)


  WOODLANDS           :  -6086 per floor *** (p = 0.0000)


  YISHUN              :  -6611 per floor *** (p = 0.0000)


## Interaction 2: remaining_lease × town

Does lease decay hurt more in less desirable towns? Losing a year of lease in Punggol might cost more (proportionally) than in Queenstown, where location holds value regardless.

In [5]:
%%R
model_int2 <- lm(resale_price ~ town * remaining_lease_years + flat_type + floor_area_sqm +
              storey_mid +
              remaining_lease_sq +
              flat_model_grouped +
              dist_cbd_km + mrt_dist_m + hawker_dist_m +
              popular_school_dist_m +
              park_dist_m + hospital_dist_m +
              columbarium_dist_m + temple_dist_m +
              coast_dist_m +
              num_eights_tail +
              price_has_168 +
              block_has_4 +
              cny_month +
              month_factor,
            data = df)

cat(sprintf('Model 10 + remaining_lease × town\n'))
cat(sprintf('R-squared:     %.4f (baseline: %.4f, improvement: %+.4f)\n',
    summary(model_int2)$r.squared, summary(model10)$r.squared,
    summary(model_int2)$r.squared - summary(model10)$r.squared))

cat('\nANOVA: does the interaction significantly improve the model?\n')
print(anova(model10, model_int2))

# Show the interaction coefficients
cat('\nInteraction coefficients (how much MORE each year of lease is worth in each town vs Ang Mo Kio):\n')
robust <- coeftest(model_int2, vcov = vcovHC(model_int2, type = 'HC1'))
int_rows <- grep('town.*:remaining_lease_years', rownames(robust))
for (i in int_rows) {
    town_name <- gsub('town|:remaining_lease_years', '', rownames(robust)[i])
    coef_val <- robust[i, 'Estimate']
    p_val <- robust[i, 'Pr(>|t|)']
    sig <- ifelse(p_val < 0.001, '***', ifelse(p_val < 0.01, '**', ifelse(p_val < 0.05, '*', '')))
    cat(sprintf('  %-20s: %+6.0f per year %s (p = %.4f)\n', town_name, coef_val, sig, p_val))
}

Model 10 + remaining_lease × town


R-squared:     0.9266 (baseline: 0.9023, improvement: +0.0243)



ANOVA: does the interaction significantly improve the model?


Analysis of Variance Table


Model 1: resale_price ~ town + flat_type + floor_area_sqm + storey_mid + 
    remaining_lease_years + remaining_lease_sq + flat_model_grouped + 
    dist_cbd_km + mrt_dist_m + hawker_dist_m + popular_school_dist_m + 
    park_dist_m + hospital_dist_m + columbarium_dist_m + temple_dist_m + 
    coast_dist_m + num_eights_tail + price_has_168 + block_has_4 + 
    cny_month + month_factor
Model 2: resale_price ~ town * remaining_lease_years + flat_type + floor_area_sqm + 
    storey_mid + remaining_lease_sq + flat_model_grouped + dist_cbd_km + 
    mrt_dist_m + hawker_dist_m + popular_school_dist_m + park_dist_m + 
    hospital_dist_m + columbarium_dist_m + temple_dist_m + coast_dist_m + 
    num_eights_tail + price_has_168 + block_has_4 + cny_month + 
    month_factor

...

  SERANGOON           :   +592 per year  (p = 0.1966)


  TAMPINES            :  -4046 per year *** (p = 0.0000)


  TOA PAYOH           :   +390 per year * (p = 0.0115)


  WOODLANDS           :  -5731 per year *** (p = 0.0000)


  YISHUN              :  -5980 per year *** (p = 0.0000)


## Interaction 3: floor_area_sqm × dist_cbd_km

Is extra space worth more in central areas? An additional sqm in Queenstown (3km from CBD) might add more to the price than in Woodlands (18km).

In [6]:
%%R
model_int3 <- lm(resale_price ~ town + flat_type + floor_area_sqm * dist_cbd_km +
              storey_mid +
              remaining_lease_years + remaining_lease_sq +
              flat_model_grouped +
              mrt_dist_m + hawker_dist_m +
              popular_school_dist_m +
              park_dist_m + hospital_dist_m +
              columbarium_dist_m + temple_dist_m +
              coast_dist_m +
              num_eights_tail +
              price_has_168 +
              block_has_4 +
              cny_month +
              month_factor,
            data = df)

cat(sprintf('Model 10 + floor_area × dist_cbd\n'))
cat(sprintf('R-squared:     %.4f (baseline: %.4f, improvement: %+.4f)\n',
    summary(model_int3)$r.squared, summary(model10)$r.squared,
    summary(model_int3)$r.squared - summary(model10)$r.squared))

cat('\nANOVA: does the interaction significantly improve the model?\n')
print(anova(model10, model_int3))

# Show the interaction coefficient
robust <- coeftest(model_int3, vcov = vcovHC(model_int3, type = 'HC1'))
int_row <- grep('floor_area_sqm:dist_cbd_km', rownames(robust))
coef_val <- robust[int_row, 'Estimate']
p_val <- robust[int_row, 'Pr(>|t|)']
cat(sprintf('\nInteraction coefficient: %+.1f (p = %.4f)\n', coef_val, p_val))
cat(sprintf('Interpretation: for every km farther from CBD, each sqm is worth $%.0f less\n', abs(coef_val)))

Model 10 + floor_area × dist_cbd


R-squared:     0.9136 (baseline: 0.9023, improvement: +0.0113)



ANOVA: does the interaction significantly improve the model?


Analysis of Variance Table


Model 1: resale_price ~ town + flat_type + floor_area_sqm + storey_mid + 
    remaining_lease_years + remaining_lease_sq + flat_model_grouped + 
    dist_cbd_km + mrt_dist_m + hawker_dist_m + popular_school_dist_m + 
    park_dist_m + hospital_dist_m + columbarium_dist_m + temple_dist_m + 
    coast_dist_m + num_eights_tail + price_has_168 + block_has_4 + 
    cny_month + month_factor
Model 2: resale_price ~ town + flat_type + floor_area_sqm * dist_cbd_km + 
    storey_mid + remaining_lease_years + remaining_lease_sq + 
    flat_model_grouped + mrt_dist_m + hawker_dist_m + popular_school_dist_m + 
    park_dist_m + hospital_dist_m + columbarium_dist_m + temple_dist_m + 
    coast_dist_m + num_eights_tail + price_has_168 + block_has_4 + 
    cny_month + month_factor

...

 ***

---
Signif. codes:  

0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


Interaction coefficient: -235.8 (p = 0.0000)


Interpretation: for every km farther from CBD, each sqm is worth $236 less


## Summary: do interactions improve the model?

In [7]:
%%R
cat(sprintf('%-35s %10s %12s\n', 'Model', 'R-squared', 'Improvement'))
cat(paste(rep('-', 60), collapse = ''), '\n')

models <- list(
    list('Model 10 (baseline)', model10),
    list('+ storey × town', model_int1),
    list('+ remaining_lease × town', model_int2),
    list('+ floor_area × dist_cbd', model_int3)
)

base_r2 <- summary(model10)$r.squared
for (m in models) {
    r2 <- summary(m[[2]])$r.squared
    imp <- r2 - base_r2
    cat(sprintf('%-35s %10.4f %+11.4f\n', m[[1]], r2, imp))
}

Model                                R-squared  Improvement


------------------------------------------------------------

Model 10 (baseline)                     0.9023     +0.0000


+ storey × town                        0.9065     +0.0042


+ remaining_lease × town               0.9266     +0.0243


+ floor_area × dist_cbd                0.9136     +0.0113


## Interpretation

All three interactions are significant (ANOVA p < 2.2e-16) and improve the model:

| Interaction | R-squared improvement |
|---|---|
| storey × town | +0.0041 |
| floor_area × dist_cbd | +0.0115 |
| **remaining_lease × town** | **+0.0239** |

### remaining_lease × town is the big finding (+2.4%)

This is the single biggest improvement since adding lease decay itself. It means lease decay is not one phenomenon — it hits very differently depending on where the flat is.

Towns where each year of lease is worth the most (vs Ang Mo Kio as baseline):
- Bukit Timah: +$20,453/year
- Marine Parade: +$14,444/year
- Bishan: +$5,450/year

Towns where each year of lease is worth the least:
- Yishun: -$5,977/year
- Jurong West: -$5,921/year
- Hougang: -$5,793/year
- Woodlands: -$5,701/year

The spread between Bukit Timah and Yishun is ~$26,000 per year of lease. Over 30 years of decay, that's a $780,000 gap.

Why: in prime towns, the location is irreplaceable — every year of lease matters because buyers are paying a huge premium to be there. In outer towns, the flat is substitutable — short lease? Just buy a newer flat nearby. Less location premium to decay.

### floor_area × dist_cbd (+1.2%)

Each sqm is worth $237 less for every km farther from CBD. Extra space is a luxury good — it commands a premium where space is scarce (central), less so where flats are already bigger (outer towns).

### storey × town (+0.4%)

Smallest effect. High floors are worth more in Marine Parade (+$3,184/floor vs AMK) and Bishan (+$1,910). Worth less in Jurong East (-$7,284), Yishun (-$6,608), and most outer towns. Views matter more where there's something to see.

### Why I didn't fold significant interaction findings into Model 10

All three interactions are statistically significant and improve the model. But I kept Model 10 without them for three reasons:

1. **Interpretability.** Model 10 has one lease decay coefficient, one storey coefficient, one floor area coefficient. Adding remaining_lease × town replaces one number with 26. "Each year of lease costs $X" becomes "each year of lease costs $X in Ang Mo Kio, $X+20,453 in Bukit Timah, $X-5,977 in Yishun..." — much harder to explain in a story or a presentation.

2. **The main findings don't change.** The superstition coefficients (8s, 168, block 4), feng shui distances (columbarium, temple), and amenity proximity (MRT, hawker, school) barely move when interactions are added. The interactions refine how we understand lease decay and floor premiums across towns, but they don't alter the other conclusions.

3. **Overfitting risk.** The three interactions together add 51 new parameters. With 50,000 observations that's still a comfortable ratio, but each additional parameter is fit to this specific 2-year window and may not generalise to other periods. Model 10 is more likely to hold up on new data.

The interactions are better presented as a separate finding: "The main model explains 90% of price variation. When I tested whether lease decay hits all towns equally, I found it doesn't — the gap between Bukit Timah and Yishun is $26,000 per year of lease."